# 介紹

本程式使用 pyautogui 來完成一個 Blitz 的遊戲機器人。

In [50]:
import pyautogui as pyg
pyg.sleep(2)
basex=107
basey=386
gap=40

In [51]:
for row in range(8):
    for col in range(8):
        x = basex + col * gap
        y = basey + row * gap
        color = pyg.pixel(x, y)
        print(row, col, x, y, color)

0 0 107 386 (253, 30, 61)
0 1 147 386 (253, 253, 253)
0 2 187 386 (184, 7, 184)
0 3 227 386 (253, 253, 121)
0 4 267 386 (81, 253, 132)
0 5 307 386 (253, 30, 61)
0 6 347 386 (253, 253, 121)
0 7 387 386 (253, 30, 61)
1 0 107 426 (253, 253, 121)
1 1 147 426 (253, 253, 39)
1 2 187 426 (184, 7, 184)
1 3 227 426 (81, 253, 132)
1 4 267 426 (235, 174, 8)
1 5 307 426 (253, 253, 121)
1 6 347 426 (81, 253, 132)
1 7 387 426 (253, 253, 39)
2 0 107 466 (253, 253, 39)
2 1 147 466 (253, 30, 61)
2 2 187 466 (253, 253, 121)
2 3 227 466 (184, 7, 184)
2 4 267 466 (22, 119, 238)
2 5 307 466 (81, 253, 132)
2 6 347 466 (184, 7, 184)
2 7 387 466 (81, 253, 132)
3 0 107 506 (253, 253, 121)
3 1 147 506 (253, 253, 39)
3 2 187 506 (184, 7, 184)
3 3 227 506 (253, 253, 121)
3 4 267 506 (184, 7, 184)
3 5 307 506 (22, 119, 238)
3 6 347 506 (81, 253, 132)
3 7 387 506 (253, 30, 61)
4 0 107 546 (253, 253, 253)
4 1 147 546 (22, 119, 238)
4 2 187 546 (253, 253, 121)
4 3 227 546 (253, 253, 121)
4 4 267 546 (81, 253, 132)
4 

In [52]:
def get_gem_position(r, c):
    """回傳第 r 列、第 c 欄寶石中心點的 (x, y) 座標。"""
    x = basex + c * gap
    y = basey + r * gap
    return (x, y)

In [ ]:
print (get_gem_position(0, 0))
print (get_gem_position(0, 1))
print (get_gem_position(1, 0))

(78, 391)
(118, 391)
(78, 431)


In [ ]:
im = pyg.screenshot()
for i in range(8):
    x, y = get_gem_position(0, i)
    pyg.click(x, y, duration=1)
    print(get_gem_color(0,i,im))

(84, 253, 136)
(22, 112, 220)
(0, 65, 171)
(0, 249, 33)
(20, 7, 29)
(20, 8, 29)
(29, 17, 40)
(165, 1, 14)


In [ ]:
def get_gem_color(r, c, im):
    """回傳截圖 im 中第 r 列、第 c 欄寶石的 RGB 顏色。"""
    x, y = get_gem_position(r, c)
    return im.getpixel((x, y))

In [58]:
def can_move_left(r, c, im):
    """判斷第 r 列、第 c 欄的寶石能否向左交換並形成三連線。"""
    if not (0 <= r < 8 and 1 <= c < 8):
        return False

    left = (r, c - 1)
    current = (r, c)
    left_color = get_gem_color(*left, im)
    current_color = get_gem_color(*current, im)
    if left_color == current_color:
        return False

    def color_after_swap(row, col):
        if (row, col) == left:
            return current_color
        if (row, col) == current:
            return left_color
        return get_gem_color(row, col, im)

    def has_match_at(row, col):
        color = color_after_swap(row, col)
        for dr, dc in ((0, 1), (1, 0)):
            count = 1
            for direction in (-1, 1):
                next_row = row + dr * direction
                next_col = col + dc * direction
                while (0 <= next_row < 8 and 0 <= next_col < 8
                       and color_after_swap(next_row, next_col) == color):
                    count += 1
                    next_row += dr * direction
                    next_col += dc * direction
            if count >= 3:
                return True
        return False

    return has_match_at(*left) or has_match_at(*current)

In [59]:
im=pyg.screenshot()
print(can_move_left(5,3,im))

False


In [ ]:
def _can_move(r, c, im, dr, dc):
    next_r, next_c = r + dr, c + dc
    if not (0 <= r < 8 and 0 <= c < 8
            and 0 <= next_r < 8 and 0 <= next_c < 8):
        return False

    current = (r, c)
    neighbor = (next_r, next_c)
    current_color = get_gem_color(r, c, im)
    neighbor_color = get_gem_color(next_r, next_c, im)
    if current_color == neighbor_color:
        return False

    def color_after_swap(row, col):
        if (row, col) == current:
            return neighbor_color
        if (row, col) == neighbor:
            return current_color
        return get_gem_color(row, col, im)

    def has_match_at(row, col):
        color = color_after_swap(row, col)
        for axis_r, axis_c in ((0, 1), (1, 0)):
            count = 1
            for direction in (-1, 1):
                check_r = row + axis_r * direction
                check_c = col + axis_c * direction
                while (0 <= check_r < 8 and 0 <= check_c < 8
                       and color_after_swap(check_r, check_c) == color):
                    count += 1
                    check_r += axis_r * direction
                    check_c += axis_c * direction
            if count >= 3:
                return True
        return False

    return has_match_at(*current) or has_match_at(*neighbor)

def can_move_right(r, c, im):
    return _can_move(r, c, im, 0, 1)

def can_move_up(r, c, im):
    return _can_move(r, c, im, -1, 0)

def can_move_down(r, c, im):
    return _can_move(r, c, im, 1, 0)